In [ ]:
from __future__ import annotations

import json
import math
import os
import shutil
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen, select_order

warnings.filterwarnings("ignore")

CONFIG = {
    # -----------------------------
    # Input files
    # -----------------------------
    "spread_panel_file": "cusip_day_spread_panel_cleaned.csv",
    "pair_shortlist_file": "pair_universe_shortlist_cross_sector.csv",
    "spread_summary_file": "cusip_day_spread_summary_cleaned.csv",
    "pair_overlap_file": "cusip_day_spread_pair_overlap_cleaned.csv",
    "validation_report_file": "validation_report.csv",

    # -----------------------------
    # Core modeling choices
    # -----------------------------
    # Use the cleaned/winsorized spread for the main model.
    # Alternative options already present in the data include daily_spread_raw,
    # daily_spread_mean_yield, daily_spread_vw_yield, etc.
    "spread_column": "daily_spread",

    # Johansen deterministic term:
    # -1 = no deterministic terms
    #  0 = constant term
    #  1 = linear trend
    # For spread relationships, allowing a constant is usually appropriate.
    "johansen_det_order": 0,

    # VECM deterministic term. "ci" places a constant inside the cointegration
    # relation, which is appropriate when the equilibrium error can have a
    # non-zero mean.
    "vecm_deterministic": "ci",

    # Lag order selection criterion for select_order(...)
    "lag_selection_ic": "aic",  # one of: aic, bic, hqic, fpe
    "max_k_ar_diff": 10,

    # Actual minimum observations required to estimate a pair, even if the
    # shortlist already filtered on overlap_days.
    "min_pair_obs": 150,

    # If True, compute bond-level ADF diagnostics once for each bond. This is
    # much cheaper than repeating level/difference unit root tests pair-by-pair.
    "run_bond_level_adf": True,
    "adf_autolag": "AIC",
    "adf_regression_levels": "c",
    "adf_regression_diffs": "c",
    "adf_regression_ect": "c",

    # Save equilibrium-error time series only for the top N pairs by trace margin.
    # Set to 0 to skip saving any pair-level time series.
    "save_top_n_ect_series": 250,

    # Reasonable half-life window used only for ranking outputs, not for hard skips.
    "ranking_half_life_min_days": 0.25,
    "ranking_half_life_max_days": 60.0,

    # -----------------------------
    # Output / runtime
    # -----------------------------
    "output_dir": "stage1_johansen_vecm_outputs",
    "checkpoint_every": 2000,
    "write_checkpoint_files": True,
    "zip_outputs": True,
    "auto_download_zip_if_colab": True,
    "random_seed": 42,
}


# =========================================================
# HELPERS
# =========================================================

def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def safe_float(x) -> float:
    try:
        return float(x)
    except Exception:
        return np.nan


def standardize_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


def bounded_maxlags(nobs: int, configured_max: int) -> int:
    """
    Conservative upper bound on k_ar_diff for pairwise estimation.
    Keeps the system estimable for shorter overlap windows.
    """
    if nobs <= 30:
        return 0
    # Heuristic: leave enough effective observations after lags.
    feasible = max(0, min(configured_max, (nobs // 8) - 1))
    return int(feasible)


def choose_k_ar_diff(data_2col: pd.DataFrame, max_k_ar_diff: int, deterministic: str, ic: str) -> Tuple[int, Dict[str, float]]:
    nobs = len(data_2col)
    maxlags = bounded_maxlags(nobs, max_k_ar_diff)
    if maxlags <= 0:
        return 0, {"selected_aic": np.nan, "selected_bic": np.nan, "selected_hqic": np.nan, "selected_fpe": np.nan}

    try:
        sel = select_order(data_2col.values, maxlags=maxlags, deterministic=deterministic)
        selected_orders = getattr(sel, "selected_orders", {}) or {}
        chosen = selected_orders.get(ic, None)
        if chosen is None or pd.isna(chosen):
            chosen = min(1, maxlags)
        chosen = int(chosen)
        diagnostics = {
            "selected_aic": safe_float(selected_orders.get("aic", np.nan)),
            "selected_bic": safe_float(selected_orders.get("bic", np.nan)),
            "selected_hqic": safe_float(selected_orders.get("hqic", np.nan)),
            "selected_fpe": safe_float(selected_orders.get("fpe", np.nan)),
        }
        return chosen, diagnostics
    except Exception:
        return 0, {"selected_aic": np.nan, "selected_bic": np.nan, "selected_hqic": np.nan, "selected_fpe": np.nan}


def adf_summary(series: pd.Series, regression: str = "c", autolag: str = "AIC") -> Dict[str, float]:
    s = pd.Series(series).dropna().astype(float)
    if len(s) < 25 or s.nunique() < 5:
        return {
            "adf_stat": np.nan,
            "adf_pvalue": np.nan,
            "usedlag": np.nan,
            "nobs": len(s),
            "crit_1pct": np.nan,
            "crit_5pct": np.nan,
            "crit_10pct": np.nan,
            "reject_5pct": np.nan,
        }
    try:
        res = adfuller(s.values, regression=regression, autolag=autolag)
        stat, pvalue, usedlag, nobs, crit, _ = res
        return {
            "adf_stat": float(stat),
            "adf_pvalue": float(pvalue),
            "usedlag": int(usedlag),
            "nobs": int(nobs),
            "crit_1pct": float(crit.get("1%", np.nan)),
            "crit_5pct": float(crit.get("5%", np.nan)),
            "crit_10pct": float(crit.get("10%", np.nan)),
            "reject_5pct": int(stat < crit.get("5%", np.nan)),
        }
    except Exception:
        return {
            "adf_stat": np.nan,
            "adf_pvalue": np.nan,
            "usedlag": np.nan,
            "nobs": len(s),
            "crit_1pct": np.nan,
            "crit_5pct": np.nan,
            "crit_10pct": np.nan,
            "reject_5pct": np.nan,
        }


def integration_classification(level_reject_5pct, diff_reject_5pct) -> str:
    if pd.isna(level_reject_5pct) or pd.isna(diff_reject_5pct):
        return "unknown"
    if (level_reject_5pct == 0) and (diff_reject_5pct == 1):
        return "I(1)-like"
    if (level_reject_5pct == 1) and (diff_reject_5pct == 1):
        return "I(0)-like"
    if (level_reject_5pct == 0) and (diff_reject_5pct == 0):
        return "higher-order-or-nonstationary"
    return "mixed"


def compute_bond_level_adf(wide: pd.DataFrame, outdir: Path) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for cusip in wide.columns:
        s = wide[cusip].dropna()
        level = adf_summary(s, regression=CONFIG["adf_regression_levels"], autolag=CONFIG["adf_autolag"])
        diff = adf_summary(s.diff().dropna(), regression=CONFIG["adf_regression_diffs"], autolag=CONFIG["adf_autolag"])
        row = {
            "cusip_id": cusip,
            "n_obs_levels": int(s.shape[0]),
            "n_obs_diffs": int(max(0, s.shape[0] - 1)),
            "level_adf_stat": level["adf_stat"],
            "level_adf_pvalue": level["adf_pvalue"],
            "level_adf_usedlag": level["usedlag"],
            "level_adf_reject_5pct": level["reject_5pct"],
            "diff_adf_stat": diff["adf_stat"],
            "diff_adf_pvalue": diff["adf_pvalue"],
            "diff_adf_usedlag": diff["usedlag"],
            "diff_adf_reject_5pct": diff["reject_5pct"],
        }
        row["integration_class_guess"] = integration_classification(
            row["level_adf_reject_5pct"], row["diff_adf_reject_5pct"]
        )
        rows.append(row)

    out = pd.DataFrame(rows).sort_values(["integration_class_guess", "cusip_id"]).reset_index(drop=True)
    out.to_csv(outdir / "bond_level_adf_diagnostics.csv", index=False)
    return out


def ols_ar1_half_life(ect: pd.Series) -> Dict[str, float]:
    e = pd.Series(ect).dropna().astype(float)
    if len(e) < 25:
        return {
            "ar1_intercept": np.nan,
            "ar1_phi": np.nan,
            "ar1_phi_tstat": np.nan,
            "ar1_phi_pvalue": np.nan,
            "ar1_r2": np.nan,
            "half_life_days": np.nan,
        }

    y = e.iloc[1:].values
    x = e.iloc[:-1].values
    X = sm.add_constant(x)

    try:
        model = OLS(y, X).fit()
        intercept = float(model.params[0])
        phi = float(model.params[1])
        tstat = float(model.tvalues[1])
        pvalue = float(model.pvalues[1])
        r2 = float(model.rsquared)
        if 0 < abs(phi) < 1:
            half_life = float(-np.log(2.0) / np.log(abs(phi)))
        else:
            half_life = np.nan
        return {
            "ar1_intercept": intercept,
            "ar1_phi": phi,
            "ar1_phi_tstat": tstat,
            "ar1_phi_pvalue": pvalue,
            "ar1_r2": r2,
            "half_life_days": half_life,
        }
    except Exception:
        return {
            "ar1_intercept": np.nan,
            "ar1_phi": np.nan,
            "ar1_phi_tstat": np.nan,
            "ar1_phi_pvalue": np.nan,
            "ar1_r2": np.nan,
            "half_life_days": np.nan,
        }


def johansen_rank_from_95(jres) -> Dict[str, object]:
    # For a bivariate system, rank can be 0 or 1 in practical use.
    trace_stat_r0 = safe_float(jres.trace_stat[0])
    trace_crit_95_r0 = safe_float(jres.trace_stat_crit_vals[0, 1])
    maxeig_stat_r0 = safe_float(jres.max_eig_stat[0])
    maxeig_crit_95_r0 = safe_float(jres.max_eig_stat_crit_vals[0, 1])

    cointegrated_trace_95 = int(trace_stat_r0 > trace_crit_95_r0)
    cointegrated_maxeig_95 = int(maxeig_stat_r0 > maxeig_crit_95_r0)
    rank_trace_95 = int(cointegrated_trace_95)
    rank_maxeig_95 = int(cointegrated_maxeig_95)

    return {
        "trace_stat_r0": trace_stat_r0,
        "trace_crit_90_r0": safe_float(jres.trace_stat_crit_vals[0, 0]),
        "trace_crit_95_r0": trace_crit_95_r0,
        "trace_crit_99_r0": safe_float(jres.trace_stat_crit_vals[0, 2]),
        "maxeig_stat_r0": maxeig_stat_r0,
        "maxeig_crit_90_r0": safe_float(jres.max_eig_stat_crit_vals[0, 0]),
        "maxeig_crit_95_r0": maxeig_crit_95_r0,
        "maxeig_crit_99_r0": safe_float(jres.max_eig_stat_crit_vals[0, 2]),
        "cointegrated_trace_95": cointegrated_trace_95,
        "cointegrated_maxeig_95": cointegrated_maxeig_95,
        "rank_trace_95": rank_trace_95,
        "rank_maxeig_95": rank_maxeig_95,
        "trace_margin_95": trace_stat_r0 - trace_crit_95_r0,
        "maxeig_margin_95": maxeig_stat_r0 - maxeig_crit_95_r0,
    }


def fit_pair_model(pair_row: pd.Series, wide: pd.DataFrame, bond_adf_lookup: Optional[pd.DataFrame]) -> Dict[str, object]:
    c1 = pair_row["cusip_1"]
    c2 = pair_row["cusip_2"]

    aligned = wide[[c1, c2]].dropna().copy()
    aligned.columns = ["y1", "y2"]
    nobs = len(aligned)

    base = {
        "cusip_1": c1,
        "cusip_2": c2,
        "company_symbol_1": pair_row.get("company_symbol_1"),
        "company_symbol_2": pair_row.get("company_symbol_2"),
        "sector_1": pair_row.get("sector_1"),
        "sector_2": pair_row.get("sector_2"),
        "sector_combo": pair_row.get("sector_combo"),
        "overlap_days_shortlist": pair_row.get("overlap_days"),
        "actual_aligned_obs": nobs,
        "spread_change_corr_shortlist": pair_row.get("spread_change_corr"),
        "avg_remaining_maturity_diff": pair_row.get("avg_remaining_maturity_diff"),
        "status": "ok",
        "skip_reason": "",
    }

    if bond_adf_lookup is not None:
        b1 = bond_adf_lookup.loc[c1] if c1 in bond_adf_lookup.index else None
        b2 = bond_adf_lookup.loc[c2] if c2 in bond_adf_lookup.index else None
        base.update({
            "bond1_integration_class_guess": None if b1 is None else b1.get("integration_class_guess"),
            "bond2_integration_class_guess": None if b2 is None else b2.get("integration_class_guess"),
            "bond1_level_adf_pvalue": None if b1 is None else b1.get("level_adf_pvalue"),
            "bond2_level_adf_pvalue": None if b2 is None else b2.get("level_adf_pvalue"),
            "bond1_diff_adf_pvalue": None if b1 is None else b1.get("diff_adf_pvalue"),
            "bond2_diff_adf_pvalue": None if b2 is None else b2.get("diff_adf_pvalue"),
        })

    if nobs < CONFIG["min_pair_obs"]:
        base.update({"status": "skipped", "skip_reason": "too_few_aligned_observations"})
        return base

    # Select lag order.
    k_ar_diff, lag_diag = choose_k_ar_diff(
        aligned,
        max_k_ar_diff=CONFIG["max_k_ar_diff"],
        deterministic=CONFIG["vecm_deterministic"],
        ic=CONFIG["lag_selection_ic"],
    )
    base.update({"k_ar_diff": k_ar_diff, **lag_diag})

    # Johansen test.
    try:
        jres = coint_johansen(aligned.values, det_order=CONFIG["johansen_det_order"], k_ar_diff=k_ar_diff)
        base.update(johansen_rank_from_95(jres))
    except Exception as e:
        base.update({"status": "failed", "skip_reason": f"johansen_error: {type(e).__name__}"})
        return base

    if base.get("rank_trace_95", 0) < 1:
        base.update({"status": "not_cointegrated_trace95"})
        return base

    # VECM estimation with rank=1.
    try:
        vecm = VECM(
            aligned.values,
            k_ar_diff=k_ar_diff,
            coint_rank=1,
            deterministic=CONFIG["vecm_deterministic"],
        )
        vres = vecm.fit()
    except Exception as e:
        base.update({"status": "failed", "skip_reason": f"vecm_error: {type(e).__name__}"})
        return base

    # Extract VECM coefficients.
    beta = np.asarray(vres.beta).reshape(-1, 1)
    alpha = np.asarray(vres.alpha).reshape(-1, 1)

    beta1 = safe_float(beta[0, 0]) if beta.shape[0] > 0 else np.nan
    beta2 = safe_float(beta[1, 0]) if beta.shape[0] > 1 else np.nan
    alpha1 = safe_float(alpha[0, 0]) if alpha.shape[0] > 0 else np.nan
    alpha2 = safe_float(alpha[1, 0]) if alpha.shape[0] > 1 else np.nan

    beta2_pvalue = np.nan
    alpha1_pvalue = np.nan
    alpha2_pvalue = np.nan

    try:
        beta2_pvalue = safe_float(vres.pvalues_beta[1, 0])
    except Exception:
        pass
    try:
        alpha1_pvalue = safe_float(vres.pvalues_alpha[0, 0])
    except Exception:
        pass
    try:
        alpha2_pvalue = safe_float(vres.pvalues_alpha[1, 0])
    except Exception:
        pass

    coint_const = 0.0
    try:
        # Constant inside the cointegration relation when deterministic="ci".
        c = np.asarray(vres.det_coef_coint)
        if c.size > 0:
            coint_const = float(c.ravel()[0])
    except Exception:
        coint_const = 0.0

    # Equilibrium error / error-correction term.
    ect = aligned.values @ beta[:, 0] + coint_const
    ect_series = pd.Series(ect, index=aligned.index, name="ect")

    ect_adf = adf_summary(
        ect_series,
        regression=CONFIG["adf_regression_ect"],
        autolag=CONFIG["adf_autolag"],
    )
    ar1 = ols_ar1_half_life(ect_series)

    # Useful hedge-ratio style transformations.
    hedge_ratio_y2_for_y1 = np.nan
    hedge_ratio_y1_for_y2 = np.nan
    if pd.notna(beta2) and abs(beta2) > 1e-12:
        hedge_ratio_y2_for_y1 = float(-beta1 / beta2)
    if pd.notna(beta1) and abs(beta1) > 1e-12:
        hedge_ratio_y1_for_y2 = float(-beta2 / beta1)

    base.update({
        "status": "vecm_estimated",
        "beta1_normalized": beta1,
        "beta2_normalized": beta2,
        "coint_const": coint_const,
        "alpha1": alpha1,
        "alpha2": alpha2,
        "alpha1_pvalue": alpha1_pvalue,
        "alpha2_pvalue": alpha2_pvalue,
        "beta2_pvalue": beta2_pvalue,
        "llf": safe_float(getattr(vres, "llf", np.nan)),
        "hedge_ratio_y2_for_y1": hedge_ratio_y2_for_y1,
        "hedge_ratio_y1_for_y2": hedge_ratio_y1_for_y2,
        "ect_mean": safe_float(ect_series.mean()),
        "ect_std": safe_float(ect_series.std(ddof=1)),
        "ect_last": safe_float(ect_series.iloc[-1]),
        "ect_last_zscore": safe_float((ect_series.iloc[-1] - ect_series.mean()) / ect_series.std(ddof=1)) if ect_series.std(ddof=1) > 0 else np.nan,
        "ect_adf_stat": ect_adf["adf_stat"],
        "ect_adf_pvalue": ect_adf["adf_pvalue"],
        "ect_adf_reject_5pct": ect_adf["reject_5pct"],
        **ar1,
    })

    # Store the time series only if requested later.
    base["_ect_series_object"] = ect_series
    return base


def write_readme(outdir: Path) -> None:
    text = f"""Stage 1 output guide
====================

Files produced by this script:

1. bond_level_adf_diagnostics.csv
   Bond-level ADF diagnostics run once per CUSIP on the cleaned spread series.
   Use this as a fast integration screen and descriptive appendix table.

2. stage1_pair_results_all.csv
   One row per shortlisted cross-sector pair. Includes skipped / failed rows too.

3. stage1_pair_results_vecm_estimated.csv
   Successful trace-cointegrated pairs where VECM(rank=1) was estimated.

4. stage1_pair_results_not_cointegrated_trace95.csv
   Pairs that failed the Johansen trace 5% threshold.

5. stage1_top_pairs_overall.csv
   High-level ranking table for economically interesting pairs.

6. stage1_top_pairs_by_sector_combo.csv
   Same idea, but balanced across sector combinations.

7. stage1_report.csv
   Run-level summary counts and timing.

8. stage1_run_config.json
   The exact configuration used.

9. ect_series_top_pairs.csv (optional)
   Saved equilibrium-error series for the top-N pairs by trace margin.

The main model choices in this script are:
- spread-based pair analysis using {CONFIG['spread_column']}
- Johansen trace test with det_order={CONFIG['johansen_det_order']}
- VECM with deterministic='{CONFIG['vecm_deterministic']}'
- lag selection by {CONFIG['lag_selection_ic']} up to max_k_ar_diff={CONFIG['max_k_ar_diff']}
- mean reversion estimated on the equilibrium error via AR(1)

"""
    (outdir / "README_outputs.txt").write_text(text)


def maybe_download_file(filepath: Path) -> None:
    if not CONFIG["auto_download_zip_if_colab"]:
        return
    try:
        from google.colab import files  # type: ignore
        files.download(str(filepath))
    except Exception:
        pass


# =========================================================
# MAIN
# =========================================================

def main() -> None:
    np.random.seed(CONFIG["random_seed"])
    t0 = time.time()

    outdir = Path(CONFIG["output_dir"])
    ensure_dir(outdir)

    print("Loading inputs...")
    spread_panel = pd.read_csv(CONFIG["spread_panel_file"], parse_dates=["trade_date", "mtrty_dt"], low_memory=False)
    spread_panel = standardize_cols(spread_panel)

    pair_shortlist = pd.read_csv(CONFIG["pair_shortlist_file"], low_memory=False)
    pair_shortlist = standardize_cols(pair_shortlist)

    # Optional files: loaded only if present.
    optional_loaded = {}
    for key in ["spread_summary_file", "pair_overlap_file", "validation_report_file"]:
        f = CONFIG.get(key)
        if f and Path(f).exists():
            try:
                optional_loaded[key] = pd.read_csv(f, low_memory=False)
            except Exception:
                optional_loaded[key] = None
        else:
            optional_loaded[key] = None

    # Build wide matrix from the cleaned long panel.
    print("Building wide spread matrix from cleaned panel...")
    wide = (
        spread_panel.pivot_table(
            index="trade_date",
            columns="cusip_id",
            values=CONFIG["spread_column"],
            aggfunc="first",
        )
        .sort_index()
        .sort_index(axis=1)
    )

    # Keep only the bonds that appear in the pair shortlist.
    shortlist_bonds = sorted(set(pair_shortlist["cusip_1"]).union(set(pair_shortlist["cusip_2"])))
    shortlist_bonds = [c for c in shortlist_bonds if c in wide.columns]
    wide = wide[shortlist_bonds].copy()

    # Bond-level ADF diagnostics.
    bond_adf = None
    if CONFIG["run_bond_level_adf"]:
        print("Running bond-level ADF diagnostics once per bond...")
        bond_adf = compute_bond_level_adf(wide, outdir)
        bond_adf_lookup = bond_adf.set_index("cusip_id")
    else:
        bond_adf_lookup = None

    # Pair loop.
    print(f"Running Johansen/VECM across {len(pair_shortlist):,} shortlisted pairs...")
    all_rows: List[Dict[str, object]] = []

    for i, (_, pair_row) in enumerate(pair_shortlist.iterrows(), start=1):
        result = fit_pair_model(pair_row, wide, bond_adf_lookup)
        all_rows.append(result)

        if (i % CONFIG["checkpoint_every"] == 0) or (i == len(pair_shortlist)):
            elapsed = time.time() - t0
            print(f"Processed {i:,}/{len(pair_shortlist):,} pairs | elapsed = {elapsed/60:.2f} min")
            if CONFIG["write_checkpoint_files"]:
                checkpoint_df = pd.DataFrame([{k: v for k, v in row.items() if not str(k).startswith("_ect_series_object")} for row in all_rows])
                checkpoint_df.to_csv(outdir / "stage1_pair_results_checkpoint.csv", index=False)

    all_df = pd.DataFrame([{k: v for k, v in row.items() if not str(k).startswith("_ect_series_object")} for row in all_rows])
    all_df.to_csv(outdir / "stage1_pair_results_all.csv", index=False)

    vecm_df = all_df.loc[all_df["status"] == "vecm_estimated"].copy()
    not_coint_df = all_df.loc[all_df["status"] == "not_cointegrated_trace95"].copy()
    failed_df = all_df.loc[all_df["status"].isin(["failed", "skipped"])].copy()

    if not vecm_df.empty:
        # Ranking diagnostics only; not a hard research filter.
        vecm_df["passes_ect_stationarity_5pct"] = (vecm_df["ect_adf_reject_5pct"] == 1).astype(int)
        vecm_df["passes_reasonable_half_life"] = (
            (vecm_df["half_life_days"] >= CONFIG["ranking_half_life_min_days"]) &
            (vecm_df["half_life_days"] <= CONFIG["ranking_half_life_max_days"])
        ).astype(int)
        vecm_df["passes_alpha_significance"] = (
            (vecm_df["alpha1_pvalue"].fillna(1.0) <= 0.05) |
            (vecm_df["alpha2_pvalue"].fillna(1.0) <= 0.05)
        ).astype(int)
        vecm_df["composite_rank_score"] = (
            vecm_df["trace_margin_95"].rank(pct=True) * 0.40 +
            vecm_df["actual_aligned_obs"].rank(pct=True) * 0.20 +
            vecm_df["passes_ect_stationarity_5pct"] * 0.20 +
            vecm_df["passes_reasonable_half_life"] * 0.10 +
            vecm_df["passes_alpha_significance"] * 0.10
        )
        vecm_df = vecm_df.sort_values(["composite_rank_score", "trace_margin_95"], ascending=[False, False])

    vecm_df.to_csv(outdir / "stage1_pair_results_vecm_estimated.csv", index=False)
    not_coint_df.to_csv(outdir / "stage1_pair_results_not_cointegrated_trace95.csv", index=False)
    failed_df.to_csv(outdir / "stage1_pair_results_failed_or_skipped.csv", index=False)

    # High-level top tables.
    if not vecm_df.empty:
        top_overall = vecm_df.head(500).copy()
        top_overall.to_csv(outdir / "stage1_top_pairs_overall.csv", index=False)

        top_by_sector_combo = (
            vecm_df.groupby("sector_combo", group_keys=False)
            .head(25)
            .copy()
        )
        top_by_sector_combo.to_csv(outdir / "stage1_top_pairs_by_sector_combo.csv", index=False)
    else:
        pd.DataFrame().to_csv(outdir / "stage1_top_pairs_overall.csv", index=False)
        pd.DataFrame().to_csv(outdir / "stage1_top_pairs_by_sector_combo.csv", index=False)

    # Optional equilibrium-error series for top N pairs.
    if CONFIG["save_top_n_ect_series"] and not vecm_df.empty:
        keep_pairs = set(
            vecm_df.sort_values(["trace_margin_95", "actual_aligned_obs"], ascending=[False, False])
            .head(CONFIG["save_top_n_ect_series"])[["cusip_1", "cusip_2"]]
            .itertuples(index=False, name=None)
        )
        ect_rows = []
        for row in all_rows:
            key = (row.get("cusip_1"), row.get("cusip_2"))
            if key in keep_pairs and ("_ect_series_object" in row):
                s = row["_ect_series_object"]
                temp = pd.DataFrame({
                    "trade_date": s.index,
                    "cusip_1": row["cusip_1"],
                    "cusip_2": row["cusip_2"],
                    "company_symbol_1": row.get("company_symbol_1"),
                    "company_symbol_2": row.get("company_symbol_2"),
                    "sector_combo": row.get("sector_combo"),
                    "ect": s.values,
                })
                ect_rows.append(temp)
        if ect_rows:
            ect_df = pd.concat(ect_rows, ignore_index=True)
            ect_df.to_csv(outdir / "ect_series_top_pairs.csv", index=False)

    # Run report.
    elapsed_sec = time.time() - t0
    report_rows = [
        {"metric": "n_shortlisted_pairs", "value": int(len(pair_shortlist))},
        {"metric": "n_unique_shortlist_bonds", "value": int(len(shortlist_bonds))},
        {"metric": "n_dates_in_wide_panel", "value": int(wide.shape[0])},
        {"metric": "n_vecm_estimated_pairs", "value": int((all_df["status"] == "vecm_estimated").sum())},
        {"metric": "n_not_cointegrated_trace95", "value": int((all_df["status"] == "not_cointegrated_trace95").sum())},
        {"metric": "n_failed_pairs", "value": int((all_df["status"] == "failed").sum())},
        {"metric": "n_skipped_pairs", "value": int((all_df["status"] == "skipped").sum())},
        {"metric": "elapsed_seconds", "value": elapsed_sec},
        {"metric": "elapsed_minutes", "value": elapsed_sec / 60.0},
        {"metric": "spread_column_used", "value": CONFIG["spread_column"]},
        {"metric": "johansen_det_order", "value": CONFIG["johansen_det_order"]},
        {"metric": "vecm_deterministic", "value": CONFIG["vecm_deterministic"]},
        {"metric": "lag_selection_ic", "value": CONFIG["lag_selection_ic"]},
        {"metric": "max_k_ar_diff", "value": CONFIG["max_k_ar_diff"]},
    ]
    report_df = pd.DataFrame(report_rows)
    report_df.to_csv(outdir / "stage1_report.csv", index=False)

    # Save config and optional validation snapshot.
    with open(outdir / "stage1_run_config.json", "w") as f:
        json.dump(CONFIG, f, indent=2)

    if optional_loaded.get("validation_report_file") is not None:
        optional_loaded["validation_report_file"].to_csv(outdir / "validation_report_snapshot.csv", index=False)

    write_readme(outdir)

    zip_path = None
    if CONFIG["zip_outputs"]:
        zip_base = outdir.parent / outdir.name
        zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=outdir)
        print(f"Zipped outputs: {zip_path}")
        maybe_download_file(Path(zip_path))

    print("Done.")
    print(f"Output folder: {outdir.resolve()}")
    if zip_path is not None:
        print(f"Zip file: {zip_path}")


if __name__ == "__main__":
    main()

Loading inputs...
Building wide spread matrix from cleaned panel...
Running bond-level ADF diagnostics once per bond...
Running Johansen/VECM across 73,777 shortlisted pairs...
Processed 2,000/73,777 pairs | elapsed = 3.19 min
Processed 4,000/73,777 pairs | elapsed = 5.83 min
Processed 6,000/73,777 pairs | elapsed = 8.51 min
Processed 8,000/73,777 pairs | elapsed = 11.13 min
Processed 10,000/73,777 pairs | elapsed = 13.80 min
Processed 12,000/73,777 pairs | elapsed = 16.35 min
Processed 14,000/73,777 pairs | elapsed = 18.93 min
Processed 16,000/73,777 pairs | elapsed = 21.51 min
Processed 18,000/73,777 pairs | elapsed = 24.09 min
Processed 20,000/73,777 pairs | elapsed = 26.68 min
Processed 22,000/73,777 pairs | elapsed = 29.32 min
Processed 24,000/73,777 pairs | elapsed = 31.87 min
Processed 26,000/73,777 pairs | elapsed = 34.46 min
Processed 28,000/73,777 pairs | elapsed = 36.94 min
Processed 30,000/73,777 pairs | elapsed = 39.46 min
Processed 32,000/73,777 pairs | elapsed = 41.98 mi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done.
Output folder: /content/stage1_johansen_vecm_outputs
Zip file: /content/stage1_johansen_vecm_outputs.zip


In [ ]:
import os
import json
import zipfile
from collections import defaultdict

import numpy as np
import pandas as pd

In [ ]:
# CONFIG
CONFIG = {
    "input_vecm_file": "stage1_pair_results_vecm_estimated.csv",
    "input_all_pairs_file": "stage1_pair_results_all.csv",  # optional, used only for context if present
    "core_half_life_min_days": 0.5,
    "core_half_life_max_days": 20.0,
    "require_both_bonds_i1_like": True,
    "require_ect_stationary_5pct": True,
    "require_alpha_significance_5pct": True,
    "require_phi_between_0_and_1": True,
    "top_balanced_issuer_cap": 5,
    "top_balanced_top_n": 500,
    "top_by_sector_issuer_cap": 3,
    "top_by_sector_max_per_sector_combo": 10,
    "output_dir": "stage1_core_filtered_outputs",
    "zip_outputs": True,
    "auto_download_zip_if_colab": True,
}

In [ ]:
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def safe_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.fillna(0).astype(float).astype(int).astype(bool)


def ranked_sort(df: pd.DataFrame) -> pd.DataFrame:
    sort_cols = []
    asc = []
    for col, direction in [
        ("composite_rank_score", False),
        ("trace_margin_95", False),
        ("actual_aligned_obs", False),
        ("spread_change_corr_shortlist", False),
        ("half_life_days", True),
    ]:
        if col in df.columns:
            sort_cols.append(col)
            asc.append(direction)
    if not sort_cols:
        return df.copy()
    return df.sort_values(sort_cols, ascending=asc).reset_index(drop=True)


def apply_issuer_cap(df: pd.DataFrame, issuer_cap: int, top_n: int | None = None) -> pd.DataFrame:
    issuer_counts = defaultdict(int)
    keep_rows = []
    for idx, row in df.iterrows():
        issuer1 = row["company_symbol_1"]
        issuer2 = row["company_symbol_2"]
        if issuer_counts[issuer1] >= issuer_cap or issuer_counts[issuer2] >= issuer_cap:
            continue
        keep_rows.append(idx)
        issuer_counts[issuer1] += 1
        issuer_counts[issuer2] += 1
        if top_n is not None and len(keep_rows) >= top_n:
            break
    return df.loc[keep_rows].copy()


def zip_directory(folder_path: str, zip_path: str) -> None:
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(folder_path):
            for file in files:
                full_path = os.path.join(root, file)
                arcname = os.path.relpath(full_path, start=os.path.dirname(folder_path))
                zf.write(full_path, arcname=arcname)


def attempt_colab_download(file_path: str) -> None:
    if not CONFIG["auto_download_zip_if_colab"]:
        return
    try:
        from google.colab import files  # type: ignore
        files.download(file_path)
        print(f"Triggered Colab download for: {file_path}")
    except Exception:
        print("Not running in Colab or automatic download unavailable. Zip file was still created.")


def main() -> None:
    output_dir = CONFIG["output_dir"]
    ensure_dir(output_dir)

    vecm = pd.read_csv(CONFIG["input_vecm_file"])
    print(f"Loaded VECM results: {vecm.shape}")

    # -----------------------------------------------------------------
    # Core filter flags
    # -----------------------------------------------------------------
    vecm["both_bonds_i1_like"] = (
        (vecm["bond1_integration_class_guess"] == "I(1)-like")
        & (vecm["bond2_integration_class_guess"] == "I(1)-like")
    )

    if "passes_ect_stationarity_5pct" in vecm.columns:
        vecm["ect_stationary_5pct"] = safe_bool(vecm["passes_ect_stationarity_5pct"])
    else:
        vecm["ect_stationary_5pct"] = safe_bool(vecm["ect_adf_reject_5pct"])

    vecm["alpha_sig_5pct"] = safe_bool(vecm["passes_alpha_significance"])
    vecm["phi_between_0_and_1"] = vecm["ar1_phi"].gt(0) & vecm["ar1_phi"].lt(1)
    vecm["half_life_in_core_window"] = vecm["half_life_days"].between(
        CONFIG["core_half_life_min_days"],
        CONFIG["core_half_life_max_days"],
        inclusive="both",
    )

    core_conditions = []
    if CONFIG["require_both_bonds_i1_like"]:
        core_conditions.append(vecm["both_bonds_i1_like"])
    if CONFIG["require_ect_stationary_5pct"]:
        core_conditions.append(vecm["ect_stationary_5pct"])
    if CONFIG["require_alpha_significance_5pct"]:
        core_conditions.append(vecm["alpha_sig_5pct"])
    if CONFIG["require_phi_between_0_and_1"]:
        core_conditions.append(vecm["phi_between_0_and_1"])
    core_conditions.append(vecm["half_life_in_core_window"])

    core_pass = np.logical_and.reduce(core_conditions)
    vecm["core_filter_pass"] = core_pass

    vecm_ranked = ranked_sort(vecm)
    core = vecm_ranked[vecm_ranked["core_filter_pass"]].copy()

    # -----------------------------------------------------------------
    # Ranked outputs
    # -----------------------------------------------------------------
    top_core_balanced = apply_issuer_cap(
        core,
        issuer_cap=CONFIG["top_balanced_issuer_cap"],
        top_n=CONFIG["top_balanced_top_n"],
    )

    by_sector_frames = []
    for sector_combo, grp in core.groupby("sector_combo", dropna=False):
        grp_ranked = ranked_sort(grp)
        grp_balanced = apply_issuer_cap(
            grp_ranked,
            issuer_cap=CONFIG["top_by_sector_issuer_cap"],
            top_n=CONFIG["top_by_sector_max_per_sector_combo"],
        )
        by_sector_frames.append(grp_balanced)

    top_core_by_sector = pd.concat(by_sector_frames, ignore_index=True) if by_sector_frames else core.head(0).copy()
    top_core_by_sector = ranked_sort(top_core_by_sector)

    # -----------------------------------------------------------------
    # Reports
    # -----------------------------------------------------------------
    unique_core_bonds = pd.unique(pd.concat([core["cusip_1"], core["cusip_2"]], ignore_index=True)).size
    unique_core_issuers = pd.unique(pd.concat([core["company_symbol_1"], core["company_symbol_2"]], ignore_index=True)).size

    report_rows = [
        ("n_vecm_pairs_input", len(vecm)),
        ("n_core_pairs", len(core)),
        ("pct_of_vecm_pairs_kept", len(core) / len(vecm) if len(vecm) else np.nan),
        ("n_sector_combos_in_core", core["sector_combo"].nunique(dropna=False)),
        ("n_unique_bonds_in_core", unique_core_bonds),
        ("n_unique_issuers_in_core", unique_core_issuers),
        ("median_half_life_core", core["half_life_days"].median() if len(core) else np.nan),
        ("mean_half_life_core", core["half_life_days"].mean() if len(core) else np.nan),
        ("median_trace_margin_core", core["trace_margin_95"].median() if len(core) else np.nan),
        ("mean_trace_margin_core", core["trace_margin_95"].mean() if len(core) else np.nan),
        ("n_top_balanced_pairs", len(top_core_balanced)),
        ("n_top_by_sector_pairs", len(top_core_by_sector)),
        ("issuer_cap_top_balanced", CONFIG["top_balanced_issuer_cap"]),
        ("issuer_cap_by_sector", CONFIG["top_by_sector_issuer_cap"]),
        ("max_pairs_per_sector_combo_by_sector_table", CONFIG["top_by_sector_max_per_sector_combo"]),
        ("core_half_life_min_days", CONFIG["core_half_life_min_days"]),
        ("core_half_life_max_days", CONFIG["core_half_life_max_days"]),
    ]
    core_report = pd.DataFrame(report_rows, columns=["metric", "value"])

    breakdown_rows = []
    for col in [
        "both_bonds_i1_like",
        "ect_stationary_5pct",
        "alpha_sig_5pct",
        "phi_between_0_and_1",
        "half_life_in_core_window",
        "core_filter_pass",
    ]:
        breakdown_rows.append({
            "filter_name": col,
            "n_pass": int(vecm[col].sum()),
            "n_total": int(len(vecm)),
            "pass_rate": float(vecm[col].mean()),
        })
    filter_breakdown = pd.DataFrame(breakdown_rows)

    # -----------------------------------------------------------------
    # Save outputs
    # -----------------------------------------------------------------
    vecm_with_flags_file = os.path.join(output_dir, "stage1_pair_results_vecm_with_core_flags.csv")
    core_pairs_file = os.path.join(output_dir, "stage1_pair_results_core_filtered.csv")
    top_balanced_file = os.path.join(output_dir, "stage1_top_pairs_core_balanced.csv")
    top_by_sector_file = os.path.join(output_dir, "stage1_top_pairs_core_by_sector_combo.csv")
    core_report_file = os.path.join(output_dir, "stage1_core_filter_report.csv")
    filter_breakdown_file = os.path.join(output_dir, "stage1_core_filter_breakdown.csv")
    config_file = os.path.join(output_dir, "stage1_core_filter_config.json")
    readme_file = os.path.join(output_dir, "README_core_filter_outputs.txt")

    vecm_ranked.to_csv(vecm_with_flags_file, index=False)
    core.to_csv(core_pairs_file, index=False)
    top_core_balanced.to_csv(top_balanced_file, index=False)
    top_core_by_sector.to_csv(top_by_sector_file, index=False)
    core_report.to_csv(core_report_file, index=False)
    filter_breakdown.to_csv(filter_breakdown_file, index=False)

    with open(config_file, "w", encoding="utf-8") as f:
        json.dump(CONFIG, f, indent=2)

    with open(readme_file, "w", encoding="utf-8") as f:
        f.write(
            "Core-filter output guide\n"
            "========================\n\n"
            "Files produced by this script:\n\n"
            "1. stage1_pair_results_vecm_with_core_flags.csv\n"
            "   Original VECM-estimated pair results with additional stricter core-filter flags.\n\n"
            "2. stage1_pair_results_core_filtered.csv\n"
            "   Pairs that pass the stricter core econometric filter:\n"
            "   - both bonds are I(1)-like\n"
            "   - ECT is stationary at 5%\n"
            "   - at least one alpha is significant at 5%\n"
            "   - AR(1) phi is between 0 and 1\n"
            "   - half-life is inside the configured core window\n\n"
            "3. stage1_top_pairs_core_balanced.csv\n"
            "   Ranked core pairs with an issuer concentration cap.\n\n"
            "4. stage1_top_pairs_core_by_sector_combo.csv\n"
            "   Ranked core pairs balanced across sector combinations.\n\n"
            "5. stage1_core_filter_report.csv\n"
            "   Summary counts and descriptive statistics for the stricter core set.\n\n"
            "6. stage1_core_filter_breakdown.csv\n"
            "   Pass rates for each individual filter and for the full core filter.\n\n"
            "7. stage1_core_filter_config.json\n"
            "   Exact configuration used.\n"
        )

    if CONFIG["zip_outputs"]:
        zip_path = os.path.join(os.getcwd(), f"{output_dir}.zip")
        zip_directory(output_dir, zip_path)
        print(f"Created zip file: {zip_path}")
        attempt_colab_download(zip_path)

    print("\nDone.")
    print(core_report)
    print("\nFilter breakdown:")
    print(filter_breakdown)

In [ ]:
main()

Loaded VECM results: (54913, 66)
Created zip file: /content/stage1_core_filtered_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Triggered Colab download for: /content/stage1_core_filtered_outputs.zip

Done.
                                        metric         value
0                           n_vecm_pairs_input  54913.000000
1                                 n_core_pairs  19614.000000
2                       pct_of_vecm_pairs_kept      0.357183
3                      n_sector_combos_in_core     55.000000
4                       n_unique_bonds_in_core    446.000000
5                     n_unique_issuers_in_core     77.000000
6                        median_half_life_core      2.603782
7                          mean_half_life_core      3.177122
8                     median_trace_margin_core      6.566981
9                       mean_trace_margin_core      8.230093
10                        n_top_balanced_pairs    175.000000
11                       n_top_by_sector_pairs    543.000000
12                     issuer_cap_top_balanced      5.000000
13                        issuer_cap_by_sector      3.000000
14  ma

In [ ]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
CONFIG = {
    "input_core_file": "stage1_pair_results_core_filtered.csv",
    "highconv_actual_aligned_obs_min": 300,
    "highconv_avg_remaining_maturity_diff_max": 1.0,
    "highconv_spread_change_corr_min": 0.40,
    "highconv_trace_margin_95_min": 5.0,
    "highconv_ect_adf_pvalue_max": 0.01,
    "highconv_half_life_min_days": 1.0,
    "highconv_half_life_max_days": 10.0,
    "top_balanced_issuer_cap": 4,
    "top_balanced_top_n": 300,
    "top_by_sector_issuer_cap": 2,
    "top_by_sector_max_per_sector_combo": 8,
    "output_dir": "stage1_high_conviction_outputs",
    "zip_outputs": True,
    "auto_download_zip_if_colab": True,
}
CONFIG

{'input_core_file': 'stage1_pair_results_core_filtered.csv',
 'highconv_actual_aligned_obs_min': 300,
 'highconv_avg_remaining_maturity_diff_max': 1.0,
 'highconv_spread_change_corr_min': 0.4,
 'highconv_trace_margin_95_min': 5.0,
 'highconv_ect_adf_pvalue_max': 0.01,
 'highconv_half_life_min_days': 1.0,
 'highconv_half_life_max_days': 10.0,
 'top_balanced_issuer_cap': 4,
 'top_balanced_top_n': 300,
 'top_by_sector_issuer_cap': 2,
 'top_by_sector_max_per_sector_combo': 8,
 'output_dir': 'stage1_high_conviction_outputs',
 'zip_outputs': True,
 'auto_download_zip_if_colab': True}

In [ ]:
def rank_norm(s: pd.Series, ascending: bool = False) -> pd.Series:
    if s.nunique(dropna=False) <= 1:
        return pd.Series(np.repeat(0.5, len(s)), index=s.index)
    return s.rank(method="average", ascending=ascending, pct=True)

def maybe_download_in_colab(path: Path, enabled: bool) -> None:
    if not enabled:
        return
    try:
        import google.colab  # type: ignore
        from google.colab import files  # type: ignore
        files.download(str(path))
    except Exception:
        pass

In [ ]:
base = Path(".")
core = pd.read_csv(base / CONFIG["input_core_file"])
core.shape, core.columns.tolist()[:20]

((19614, 72),
 ['cusip_1',
  'cusip_2',
  'company_symbol_1',
  'company_symbol_2',
  'sector_1',
  'sector_2',
  'sector_combo',
  'overlap_days_shortlist',
  'actual_aligned_obs',
  'spread_change_corr_shortlist',
  'avg_remaining_maturity_diff',
  'status',
  'skip_reason',
  'bond1_integration_class_guess',
  'bond2_integration_class_guess',
  'bond1_level_adf_pvalue',
  'bond2_level_adf_pvalue',
  'bond1_diff_adf_pvalue',
  'bond2_diff_adf_pvalue',
  'k_ar_diff'])

In [ ]:
required = [
    "actual_aligned_obs",
    "avg_remaining_maturity_diff",
    "spread_change_corr_shortlist",
    "trace_margin_95",
    "ect_adf_pvalue",
    "half_life_days",
    "company_symbol_1",
    "company_symbol_2",
    "cusip_1",
    "cusip_2",
    "sector_combo",
]
missing = [c for c in required if c not in core.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = core.copy()

df["passes_hc_overlap"] = df["actual_aligned_obs"] >= CONFIG["highconv_actual_aligned_obs_min"]
df["passes_hc_maturity"] = df["avg_remaining_maturity_diff"] <= CONFIG["highconv_avg_remaining_maturity_diff_max"]
df["passes_hc_corr"] = df["spread_change_corr_shortlist"] >= CONFIG["highconv_spread_change_corr_min"]
df["passes_hc_trace_margin"] = df["trace_margin_95"] >= CONFIG["highconv_trace_margin_95_min"]
df["passes_hc_ect_strict"] = df["ect_adf_pvalue"] <= CONFIG["highconv_ect_adf_pvalue_max"]
df["passes_hc_half_life"] = df["half_life_days"].between(
    CONFIG["highconv_half_life_min_days"],
    CONFIG["highconv_half_life_max_days"],
    inclusive="both",
)

hc_cols = [c for c in df.columns if c.startswith("passes_hc_")]
df["high_conviction_pass"] = df[hc_cols].all(axis=1)

hc = df[df["high_conviction_pass"]].copy()
len(hc), round(100 * len(hc) / len(df), 2)

(2543, 12.97)

In [ ]:
hc["rank_trace"] = rank_norm(hc["trace_margin_95"], ascending=False)
hc["rank_corr"] = rank_norm(hc["spread_change_corr_shortlist"], ascending=False)
hc["rank_overlap"] = rank_norm(hc["actual_aligned_obs"], ascending=False)
hc["rank_maturity"] = rank_norm(hc["avg_remaining_maturity_diff"], ascending=True)
hc["rank_ect"] = rank_norm(hc["ect_adf_pvalue"], ascending=True)
hc["hl_distance_from_3"] = (hc["half_life_days"] - 3.0).abs()
hc["rank_half_life_shape"] = rank_norm(hc["hl_distance_from_3"], ascending=True)

hc["high_conviction_rank_score"] = (
    0.25 * hc["rank_trace"]
    + 0.20 * hc["rank_corr"]
    + 0.20 * hc["rank_overlap"]
    + 0.15 * hc["rank_maturity"]
    + 0.10 * hc["rank_ect"]
    + 0.10 * hc["rank_half_life_shape"]
)

hc = hc.sort_values(
    ["high_conviction_rank_score", "trace_margin_95"],
    ascending=[False, False],
).reset_index(drop=True)

hc[[
    "company_symbol_1","company_symbol_2","sector_combo","actual_aligned_obs",
    "avg_remaining_maturity_diff","spread_change_corr_shortlist",
    "trace_margin_95","half_life_days","high_conviction_rank_score"
]].head(10)

,company_symbol_1,company_symbol_2,sector_combo,actual_aligned_obs,avg_remaining_maturity_diff,spread_change_corr_shortlist,trace_margin_95,half_life_days,high_conviction_rank_score
0,ATO,DE,Industrials | Utilities,422,0.997385,0.430497,5.913874,1.564870,0.847523
1,AZO,CAT,Consumer Discretionary | Industrials,453,0.986337,0.405836,5.240412,3.655278,0.846638
2,CAT,CNP,Industrials | Utilities,301,0.846106,0.426567,7.020853,1.862181,0.841663
3,AMT,GE,Industrials | Real Estate,418,0.813472,0.437161,5.868346,4.381877,0.841368
4,AEE,DVN,Energy | Utilities,437,0.675988,0.438633,5.168855,3.914487,0.828234
5,CRH,FANG,Energy | Materials,341,0.708413,0.460800,5.059885,3.720548,0.825501
6,BK,CNP,Financials | Utilities,417,0.863464,0.434045,5.636376,2.662305,0.822847
7,CRH,FANG,Energy | Materials,343,0.692752,0.478066,6.098033,5.968912,0.820468
8,AEP,ADM,Consumer Staples | Utilities,429,0.924534,0.420145,5.413482,3.504832,0.810676
9,AEP,BA,Industrials | Utilities,453,0.916810,0.408088,8.206276,5.406630,0.807177


In [ ]:
rows = []
issuer_counts = {}
for _, row in hc.iterrows():
    i1 = row["company_symbol_1"]
    i2 = row["company_symbol_2"]
    if issuer_counts.get(i1, 0) >= CONFIG["top_balanced_issuer_cap"]:
        continue
    if issuer_counts.get(i2, 0) >= CONFIG["top_balanced_issuer_cap"]:
        continue
    rows.append(row)
    issuer_counts[i1] = issuer_counts.get(i1, 0) + 1
    issuer_counts[i2] = issuer_counts.get(i2, 0) + 1
    if len(rows) >= CONFIG["top_balanced_top_n"]:
        break
top_balanced = pd.DataFrame(rows)

rows = []
issuer_combo_counts = {}
combo_counts = {}
for _, row in hc.iterrows():
    combo = row["sector_combo"]
    i1 = row["company_symbol_1"]
    i2 = row["company_symbol_2"]
    if combo_counts.get(combo, 0) >= CONFIG["top_by_sector_max_per_sector_combo"]:
        continue
    if issuer_combo_counts.get((combo, i1), 0) >= CONFIG["top_by_sector_issuer_cap"]:
        continue
    if issuer_combo_counts.get((combo, i2), 0) >= CONFIG["top_by_sector_issuer_cap"]:
        continue
    rows.append(row)
    combo_counts[combo] = combo_counts.get(combo, 0) + 1
    issuer_combo_counts[(combo, i1)] = issuer_combo_counts.get((combo, i1), 0) + 1
    issuer_combo_counts[(combo, i2)] = issuer_combo_counts.get((combo, i2), 0) + 1
top_by_sector = pd.DataFrame(rows)

len(top_balanced), len(top_by_sector)

(119, 341)

In [ ]:
report = {
    "core_pairs_input": int(len(df)),
    "high_conviction_pairs_kept": int(len(hc)),
    "kept_share_of_core_pct": round(100 * len(hc) / len(df), 2) if len(df) else 0.0,
    "unique_bonds_in_high_conviction": int(len(set(hc["cusip_1"]).union(hc["cusip_2"]))),
    "unique_issuers_in_high_conviction": int(len(set(hc["company_symbol_1"]).union(hc["company_symbol_2"]))),
    "sector_combos_in_high_conviction": int(hc["sector_combo"].nunique()),
    "median_half_life_days": float(hc["half_life_days"].median()) if len(hc) else np.nan,
    "mean_half_life_days": float(hc["half_life_days"].mean()) if len(hc) else np.nan,
    "median_trace_margin_95": float(hc["trace_margin_95"].median()) if len(hc) else np.nan,
    "mean_trace_margin_95": float(hc["trace_margin_95"].mean()) if len(hc) else np.nan,
    "median_actual_aligned_obs": float(hc["actual_aligned_obs"].median()) if len(hc) else np.nan,
    "mean_actual_aligned_obs": float(hc["actual_aligned_obs"].mean()) if len(hc) else np.nan,
    "median_spread_change_corr": float(hc["spread_change_corr_shortlist"].median()) if len(hc) else np.nan,
    "mean_spread_change_corr": float(hc["spread_change_corr_shortlist"].mean()) if len(hc) else np.nan,
    "median_avg_remaining_maturity_diff": float(hc["avg_remaining_maturity_diff"].median()) if len(hc) else np.nan,
    "mean_avg_remaining_maturity_diff": float(hc["avg_remaining_maturity_diff"].mean()) if len(hc) else np.nan,
    "balanced_top_pairs_rows": int(len(top_balanced)),
    "by_sector_top_pairs_rows": int(len(top_by_sector)),
}
report_df = pd.DataFrame(list(report.items()), columns=["metric", "value"])
report_df

,metric,value
0,core_pairs_input,19614.000000
1,high_conviction_pairs_kept,2543.000000
2,kept_share_of_core_pct,12.970000
3,unique_bonds_in_high_conviction,316.000000
4,unique_issuers_in_high_conviction,68.000000
5,sector_combos_in_high_conviction,54.000000
6,median_half_life_days,2.914565
7,mean_half_life_days,3.226378
8,median_trace_margin_95,9.917271
9,mean_trace_margin_95,11.297975


In [ ]:
breakdown_rows = []
for col in hc_cols:
    breakdown_rows.append(
        {
            "filter": col,
            "pass_count": int(df[col].sum()),
            "pass_rate_pct": round(100 * df[col].mean(), 2),
        }
    )
breakdown_rows.append(
    {
        "filter": "high_conviction_pass",
        "pass_count": int(df["high_conviction_pass"].sum()),
        "pass_rate_pct": round(100 * df["high_conviction_pass"].mean(), 2),
    }
)
breakdown_df = pd.DataFrame(breakdown_rows)
breakdown_df

,filter,pass_count,pass_rate_pct
0,passes_hc_overlap,13980,71.28
1,passes_hc_maturity,10421,53.13
2,passes_hc_corr,13311,67.86
3,passes_hc_trace_margin,12027,61.32
4,passes_hc_ect_strict,14588,74.38
5,passes_hc_half_life,18226,92.92
6,high_conviction_pass,2543,12.97


In [ ]:
outdir = base / CONFIG["output_dir"]
outdir.mkdir(parents=True, exist_ok=True)

df.to_csv(outdir / "stage1_pair_results_core_with_high_conviction_flags.csv", index=False)
hc.to_csv(outdir / "stage1_pair_results_high_conviction.csv", index=False)
top_balanced.to_csv(outdir / "stage1_top_pairs_high_conviction_balanced.csv", index=False)
top_by_sector.to_csv(outdir / "stage1_top_pairs_high_conviction_by_sector_combo.csv", index=False)
report_df.to_csv(outdir / "stage1_high_conviction_filter_report.csv", index=False)
breakdown_df.to_csv(outdir / "stage1_high_conviction_filter_breakdown.csv", index=False)
with open(outdir / "stage1_high_conviction_filter_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

with open(outdir / "README_high_conviction_outputs.txt", "w") as f:
    f.write(
        "High-conviction filter output guide\n"
        "===================================\n\n"
        "Files produced by this script:\n\n"
        "1. stage1_pair_results_core_with_high_conviction_flags.csv\n"
        "   Core-filtered pairs with additional high-conviction filter flags.\n\n"
        "2. stage1_pair_results_high_conviction.csv\n"
        "   Pairs that pass the tighter final economic screen:\n"
        "   - actual_aligned_obs >= configured minimum\n"
        "   - avg_remaining_maturity_diff <= configured maximum\n"
        "   - spread_change_corr_shortlist >= configured minimum\n"
        "   - trace_margin_95 >= configured minimum\n"
        "   - ect_adf_pvalue <= configured maximum\n"
        "   - half_life_days inside the configured final window\n\n"
        "3. stage1_top_pairs_high_conviction_balanced.csv\n"
        "   Ranked high-conviction pairs with an issuer concentration cap.\n\n"
        "4. stage1_top_pairs_high_conviction_by_sector_combo.csv\n"
        "   Ranked high-conviction pairs balanced across sector combinations.\n\n"
        "5. stage1_high_conviction_filter_report.csv\n"
        "   Summary counts and descriptive statistics for the final high-conviction set.\n\n"
        "6. stage1_high_conviction_filter_breakdown.csv\n"
        "   Pass rates for each individual high-conviction filter and for the full filter.\n\n"
        "7. stage1_high_conviction_filter_config.json\n"
        "   Exact configuration used.\n"
    )

if CONFIG["zip_outputs"]:
    zip_path = base / "stage1_high_conviction_outputs.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in outdir.iterdir():
            zf.write(p, arcname=f"{outdir.name}/{p.name}")
    maybe_download_in_colab(zip_path, CONFIG["auto_download_zip_if_colab"])

print(f"Saved outputs to: {outdir.resolve()}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved outputs to: /content/stage1_high_conviction_outputs


In [ ]:
import json, os, zipfile
import numpy as np
import pandas as pd

CONFIG = {
    "input_high_conviction_file": "stage1_pair_results_high_conviction.csv",
    "target_n_pairs": 1000,
    "reserve_n_pairs": 150,
    "guaranteed_per_sector_combo": 8,
    "issuer_cap_total": 48,
    "issuer_cap_per_sector_combo": 8,
    "reserve_extra_issuer_cap_total": 15,
    "reserve_extra_issuer_cap_per_sector_combo": 4,
    "prefer_half_life_target_days": 3.0,
    "ranking_score_column": "high_conviction_rank_score",
    "output_dir": "stage2_regime_switching_selector_outputs",
    "zip_outputs": True,
    "auto_download_zip_if_colab": True,
}
CONFIG

{'input_high_conviction_file': 'stage1_pair_results_high_conviction.csv',
 'target_n_pairs': 1000,
 'reserve_n_pairs': 150,
 'guaranteed_per_sector_combo': 8,
 'issuer_cap_total': 48,
 'issuer_cap_per_sector_combo': 8,
 'reserve_extra_issuer_cap_total': 15,
 'reserve_extra_issuer_cap_per_sector_combo': 4,
 'prefer_half_life_target_days': 3.0,
 'ranking_score_column': 'high_conviction_rank_score',
 'output_dir': 'stage2_regime_switching_selector_outputs',
 'zip_outputs': True,
 'auto_download_zip_if_colab': True}

In [ ]:
def build_stage2_score(df: pd.DataFrame, half_life_target: float) -> pd.DataFrame:
    df = df.copy()
    df["hl_target_distance"] = (df["half_life_days"] - half_life_target).abs()
    df["score_trace"] = df["trace_margin_95"].rank(pct=True)
    df["score_ect"] = 1 - df["ect_adf_pvalue"].rank(pct=True)
    df["score_overlap"] = df["actual_aligned_obs"].rank(pct=True)
    df["score_corr"] = df["spread_change_corr_shortlist"].rank(pct=True)
    df["score_maturity"] = 1 - df["avg_remaining_maturity_diff"].rank(pct=True)
    df["score_hl_target"] = 1 - df["hl_target_distance"].rank(pct=True)

    df["stage2_regime_score"] = (
        0.30 * df["high_conviction_rank_score"].rank(pct=True)
        + 0.20 * df["score_trace"]
        + 0.15 * df["score_ect"]
        + 0.15 * df["score_overlap"]
        + 0.10 * df["score_corr"]
        + 0.05 * df["score_maturity"]
        + 0.05 * df["score_hl_target"]
    )
    return df

In [ ]:
df = pd.read_csv(CONFIG["input_high_conviction_file"])
df = build_stage2_score(df, CONFIG["prefer_half_life_target_days"])
df = df.sort_values(
    ["stage2_regime_score", "high_conviction_rank_score", "trace_margin_95", "actual_aligned_obs"],
    ascending=False
).reset_index(drop=True)
df.shape

(2543, 95)

In [ ]:
issuer_counts = {}
sector_issuer_counts = {}
selected_idx = []
selection_stage = []
selected_set = set()

def can_add_row(row, extra_total=0, extra_combo=0):
    combo = row["sector_combo"]
    for issuer in [row["company_symbol_1"], row["company_symbol_2"]]:
        if issuer_counts.get(issuer, 0) >= CONFIG["issuer_cap_total"] + extra_total:
            return False
        if sector_issuer_counts.get((combo, issuer), 0) >= CONFIG["issuer_cap_per_sector_combo"] + extra_combo:
            return False
    return True

def add_row(idx, stage):
    row = df.loc[idx]
    combo = row["sector_combo"]
    selected_idx.append(idx)
    selection_stage.append(stage)
    selected_set.add(idx)
    for issuer in [row["company_symbol_1"], row["company_symbol_2"]]:
        issuer_counts[issuer] = issuer_counts.get(issuer, 0) + 1
        sector_issuer_counts[(combo, issuer)] = sector_issuer_counts.get((combo, issuer), 0) + 1

for combo, sub in df.groupby("sector_combo", sort=False):
    count = 0
    for idx in sub.index:
        if len(selected_idx) >= CONFIG["target_n_pairs"]:
            break
        if count >= CONFIG["guaranteed_per_sector_combo"]:
            break
        if can_add_row(df.loc[idx]):
            add_row(idx, "guaranteed_sector_combo")
            count += 1

for idx in df.index:
    if len(selected_idx) >= CONFIG["target_n_pairs"]:
        break
    if idx in selected_set:
        continue
    if can_add_row(df.loc[idx]):
        add_row(idx, "global_fill")

selected = df.loc[selected_idx[:CONFIG["target_n_pairs"]]].copy()
selected["selection_stage"] = selection_stage[:CONFIG["target_n_pairs"]]
selected = selected.sort_values(
    ["stage2_regime_score", "high_conviction_rank_score"],
    ascending=False
).reset_index(drop=True)
selected["regime_input_rank"] = np.arange(1, len(selected) + 1)

selected.shape, selected["sector_combo"].nunique()

((1000, 97), 54)

In [ ]:
issuer_counts = {}
sector_issuer_counts = {}
for _, row in selected.iterrows():
    combo = row["sector_combo"]
    for issuer in [row["company_symbol_1"], row["company_symbol_2"]]:
        issuer_counts[issuer] = issuer_counts.get(issuer, 0) + 1
        sector_issuer_counts[(combo, issuer)] = sector_issuer_counts.get((combo, issuer), 0) + 1

selected_pairs = set(zip(selected["cusip_1"], selected["cusip_2"]))

reserve_rows = []
for idx, row in df.iterrows():
    pair = (row["cusip_1"], row["cusip_2"])
    if pair in selected_pairs:
        continue
    if can_add_row(
        row,
        CONFIG["reserve_extra_issuer_cap_total"],
        CONFIG["reserve_extra_issuer_cap_per_sector_combo"]
    ):
        reserve_rows.append(idx)
        combo = row["sector_combo"]
        for issuer in [row["company_symbol_1"], row["company_symbol_2"]]:
            issuer_counts[issuer] = issuer_counts.get(issuer, 0) + 1
            sector_issuer_counts[(combo, issuer)] = sector_issuer_counts.get((combo, issuer), 0) + 1
        if len(reserve_rows) >= CONFIG["reserve_n_pairs"]:
            break

reserve = df.loc[reserve_rows].copy()
reserve["selection_stage"] = "reserve_fill"
reserve = reserve.sort_values(
    ["stage2_regime_score", "high_conviction_rank_score"],
    ascending=False
).reset_index(drop=True)
reserve["reserve_rank"] = np.arange(1, len(reserve) + 1)

reserve.shape

(150, 97)

In [ ]:
output_dir = CONFIG["output_dir"]
os.makedirs(output_dir, exist_ok=True)

selected.to_csv(os.path.join(output_dir, "stage2_regime_switching_input.csv"), index=False)
reserve.to_csv(os.path.join(output_dir, "stage2_regime_switching_reserve.csv"), index=False)

report = pd.DataFrame([
    ("input_high_conviction_pairs", len(df)),
    ("target_n_pairs", CONFIG["target_n_pairs"]),
    ("selected_n_pairs", len(selected)),
    ("reserve_n_pairs_target", CONFIG["reserve_n_pairs"]),
    ("reserve_n_pairs_selected", len(reserve)),
    ("guaranteed_per_sector_combo", CONFIG["guaranteed_per_sector_combo"]),
    ("issuer_cap_total", CONFIG["issuer_cap_total"]),
    ("issuer_cap_per_sector_combo", CONFIG["issuer_cap_per_sector_combo"]),
    ("reserve_extra_issuer_cap_total", CONFIG["reserve_extra_issuer_cap_total"]),
    ("reserve_extra_issuer_cap_per_sector_combo", CONFIG["reserve_extra_issuer_cap_per_sector_combo"]),
    ("unique_sector_combos_input", df["sector_combo"].nunique()),
    ("unique_sector_combos_selected", selected["sector_combo"].nunique()),
    ("unique_issuers_selected", len(set(selected["company_symbol_1"]).union(set(selected["company_symbol_2"])))),
    ("unique_bonds_selected", len(set(selected["cusip_1"]).union(set(selected["cusip_2"])))),
    ("median_half_life_selected", float(selected["half_life_days"].median())),
    ("mean_half_life_selected", float(selected["half_life_days"].mean())),
    ("median_trace_margin_selected", float(selected["trace_margin_95"].median())),
    ("median_overlap_selected", float(selected["actual_aligned_obs"].median())),
    ("median_corr_selected", float(selected["spread_change_corr_shortlist"].median())),
    ("median_maturity_diff_selected", float(selected["avg_remaining_maturity_diff"].median())),
], columns=["metric", "value"])
report.to_csv(os.path.join(output_dir, "stage2_regime_switching_selector_report.csv"), index=False)

issuer_use = pd.concat([
    selected[["company_symbol_1"]].rename(columns={"company_symbol_1": "issuer"}),
    selected[["company_symbol_2"]].rename(columns={"company_symbol_2": "issuer"}),
]).value_counts().rename("pair_appearances").reset_index()
issuer_use.to_csv(os.path.join(output_dir, "stage2_regime_switching_issuer_usage.csv"), index=False)

sector_counts = selected["sector_combo"].value_counts().rename_axis("sector_combo").reset_index(name="selected_pair_count")
sector_counts.to_csv(os.path.join(output_dir, "stage2_regime_switching_sector_combo_counts.csv"), index=False)

breakdown = selected["selection_stage"].value_counts().rename_axis("selection_stage").reset_index(name="count")
breakdown.to_csv(os.path.join(output_dir, "stage2_regime_switching_selector_breakdown.csv"), index=False)

with open(os.path.join(output_dir, "stage2_regime_switching_selector_config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)

report

,metric,value
0,input_high_conviction_pairs,2543.000000
1,target_n_pairs,1000.000000
2,selected_n_pairs,1000.000000
3,reserve_n_pairs_target,150.000000
4,reserve_n_pairs_selected,150.000000
5,guaranteed_per_sector_combo,8.000000
6,issuer_cap_total,48.000000
7,issuer_cap_per_sector_combo,8.000000
8,reserve_extra_issuer_cap_total,15.000000
9,reserve_extra_issuer_cap_per_sector_combo,4.000000


In [ ]:
if CONFIG["zip_outputs"]:
    zip_path = output_dir + ".zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(output_dir):
            for file in files:
                full = os.path.join(root, file)
                arc = os.path.relpath(full, output_dir)
                zf.write(full, arc)
    print("Wrote:", zip_path)

    if CONFIG["auto_download_zip_if_colab"]:
        try:
            from google.colab import files  # type: ignore
            files.download(zip_path)
        except Exception:
            pass

Wrote: stage2_regime_switching_selector_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>